# 3장 1강: LLM의 기본 개념과 특성 — 실습문제


## 실습 목표

- TF-IDF, 텍스트 임베딩, LLM의 주요 역할을 구분할 수 있다.
- LLM의 토큰·문맥·다음 토큰 예측과 텍스트 생성 흐름을 설명할 수 있다.
- 같은 상담 원문도 작업 지시에 따라 분류·요약·정보 추출 결과가 달라짐을 설명할 수 있다.
- 출력의 표현 차이, 근거 없는 정보, 원문 왜곡, 분류 오류를 구분할 수 있다.
- 입력과 출력의 연결, 결과 누락, 입력 크기 제한과 개인정보 관련 주의점을 확인할 수 있다.

## 실습 환경 / 데이터

- Python 3, Jupyter Notebook 또는 VS Code 노트북, pandas
- 아래 CSV 3개를 노트북과 같은 폴더에 둡니다. 다른 폴더라면 `DATA_DIR`를 수정합니다.
- 앞 강의의 실행 결과는 필요하지 않습니다. 준비 코드부터 위에서 아래로 실행합니다.

| 파일 | 이번 실습의 역할 |
|---|---|
| 엘지유플러스_분류.csv | 상담을 정해진 범주로 구분하는 지시와 저장된 출력 확인 |
| 엘지유플러스_요약.csv | 상담 내용을 간결하게 정리하는 지시와 저장된 출력 확인 |
| 엘지유플러스_질의응답.csv | 상담의 특정 사실을 묻는 질문과 저장된 답변 확인 |

| 공통 컬럼 | 의미 |
|---|---|
| source_id | 상담 원문을 구분하는 ID. 같은 상담으로 여러 작업을 만들면 반복됨 |
| consulting_category | 데이터에 부여된 상담 분야 정보 |
| task_category | 상담 내용 분류, 기본 요약, 의문사형 등 세부 작업 유형 |
| instruction | 해당 행에서 요청하는 작업 지시 또는 질문 |
| input | 분석할 상담 원문 |
| output | 해당 행에 저장되어 있는 출력. 이번 노트북에서 새로 생성한 응답이 아님 |

### 실습 범위와 데이터 구분

교안의 리뷰 분석 사례를 **상담 텍스트 분석**에 적용합니다. `input`과 `instruction`을 읽고, 어떤 출력이 필요한지와 결과가 원문에 맞는지를 확인합니다.

- **실제 데이터:** CSV에 저장된 원문·지시·출력·ID입니다. 저장된 `output`도 원문과 비교할 참고 자료이며, 무조건 정확하다고 가정하지 않습니다.
- **학습용 예시:** 필수 2와 과제에 제공하는 검토 대상 출력은 작성자가 만든 가상 예시입니다. 실제 LLM을 여러 번 호출해 얻은 결과가 아닙니다.
- **정답 코드의 역할:** CSV 조회, 표 구성, 범주 문자열 확인, 누락 ID 확인입니다. 의미의 정확성은 사람이 원문을 읽고 판단합니다.
- 실제 LLM/API 호출, 모델 학습, Transformer 내부 계산, 상세 프롬프트 설계, 자동 평가 지표 계산은 요구하지 않습니다. 교안의 개념과 검증 관점에 집중합니다.
- 상담 속 요금·서비스 안내는 해당 원문을 해석하기 위한 자료입니다. 현재 상품 정책이나 모든 고객에게 적용되는 사실로 확대하지 않습니다.
- `<NAME>`, `<DATE>`, `<CHARGE>` 같은 표시는 원문에 있는 가림 표시입니다. 이름·날짜·금액을 추측해 채우지 않습니다.

**난이도:** 기초~중급. 필수 1·필수 2·과제 1로 구성하며, 과제도 필수 문제와 같은 수준의 읽기·비교·검증을 수행합니다.


## 시작 전에 읽는 개념 정리

### 1. 이번에는 ‘어떤 문서를 찾을까’에서 ‘내용을 어떻게 분석할까’로 이동합니다

앞 강의에서는 의미가 비슷한 문서를 찾고, 검색 결과가 질문에 맞는지 확인했습니다. 이번에는 상담 원문을 읽고 **분류·요약·특정 사실 확인**처럼 목적이 다른 작업을 구분합니다. 같은 원문을 쓰더라도 지시가 달라지면 필요한 출력도 달라집니다.

| 방법 | 주요 역할 | 상담 데이터에 적용하면 |
|---|---|---|
| TF-IDF | 문서에서 특징적인 단어의 중요도 계산 | 어떤 단어가 문서를 특징짓는지 살펴봄 |
| 텍스트 임베딩 | 의미적 특징을 벡터로 표현 | 질문과 유사한 상담을 검색할 때 활용 |
| LLM | 문맥과 작업 지시를 바탕으로 결과 생성 | 상담을 분류하거나 주요 내용을 요약하도록 요청 |

이 표는 각 방법의 주요 역할을 비교한 것입니다. 분류는 LLM만 할 수 있다거나, 세 방법을 함께 사용할 수 없다는 뜻은 아닙니다. 이번에는 교안에서 다룬 LLM의 동작 특성과 결과 검증에 집중합니다.

### 2. 처음 나오는 용어를 쉬운 말로 정리하기

| 용어 | 쉬운 설명 | 읽을 때 주의할 점 |
|---|---|---|
| LLM | 많은 언어 데이터를 학습하여 텍스트 처리와 생성에 활용하는 모델 | 자연스러운 응답과 사실의 정확성은 구분 |
| 토큰 | 모델이 텍스트를 처리하는 기본 단위 | 단어·단어 일부·문장부호 등이 될 수 있으며 글자 수와 같지 않음 |
| 문맥 | 응답을 만들 때 참고하는 입력과 앞서 생성한 내용 | 상담의 앞뒤 발화를 함께 읽는 이유와 연결 |
| 다음 토큰 예측 | 앞선 문맥에서 다음에 올 토큰의 확률을 예측하는 과정 | 무작위로 아무 단어나 붙인다는 뜻은 아님 |
| 입력 | 분석할 본문, 지시, 참고 기준 등 전달하는 정보 | CSV의 `input` 열은 그중 상담 본문에 해당 |
| 출력 | 요청한 작업에 대한 결과 | CSV의 `output`을 읽는 것과 모델이 새로 생성하는 것은 다름 |
| 분류 | 정해진 범주 중 어디에 속하는지 구분 | 범주명과 실제 내용이 모두 맞아야 함 |
| 요약 | 핵심 내용을 짧게 정리 | 짧아져도 사실이나 조건을 바꾸면 안 됨 |
| 정보 추출 | 필요한 항목을 원문에서 찾아 정리 | 원문에 없는 항목은 추측으로 채우지 않음 |
| 출력 변동성 | 같은 작업에서도 표현이나 결과가 달라질 수 있는 특성 | 표현 차이가 모두 오류는 아니며 내용 확인이 필요 |
| 환각 | 근거가 없거나 사실과 다른 내용을 생성하는 현상 | 그럴듯한 문장도 원문과 대조 |
| 정확성 | 결과가 원문의 사실과 작업 기준에 맞는 정도 | 이번에는 점수 계산 대신 사례의 의미를 판단 |
| 출력 일관성 | 같은 기준에 따라 결과를 일관되게 표현하는 것 | 틀린 답을 일관되게 내는 경우도 가능 |
| 컨텍스트 윈도우 | 한 번의 처리에서 다룰 수 있는 토큰 범위 | 많은 상담을 무제한으로 입력할 수는 없음 |

### 3. LLM이 응답을 만드는 흐름을 단계별로 이해하기

| 단계 | 일어나는 일 | 핵심 의미 |
|---|---|---|
| 1. 입력 확인 | 본문·작업 지시·참고 기준을 전달 | 무엇을 분석하고 어떤 결과를 원하는지 알려 줌 |
| 2. 토큰 단위 처리 | 텍스트를 모델이 처리하는 단위로 다룸 | 사람이 띄어 쓴 단어와 항상 일치하지 않음 |
| 3. 다음 토큰 예측 | 입력과 지금까지의 생성 내용을 바탕으로 다음 토큰의 확률을 예측 | 앞선 문맥이 이어질 내용에 영향을 줌 |
| 4. 토큰 생성 | 다음 토큰을 생성하여 응답에 추가 | 만들어진 내용도 이후의 문맥이 됨 |
| 5. 반복 | 다음 토큰을 다시 예측하고 생성 | 여러 단계를 거쳐 응답이 완성됨 |

이 흐름은 **LLM의 생성 원리 설명**입니다. 이 노트북에는 토큰을 나누는 모델이나 다음 토큰을 예측하는 코드가 없습니다. CSV의 저장된 결과를 읽고, 사람이 만든 예시의 정확성을 검토합니다. 저장된 답변을 조회했다고 LLM을 실행한 것으로 해석하지 마세요.

### 4. 세 종류의 자료를 구분하고 시작하기

| 자료 | 어디에서 왔나요? | 이번 실습에서 하는 일 |
|---|---|---|
| 원문·지시·저장된 출력 | 제공된 CSV | 조회하고 서로 비교 |
| A~D, F1~F3, T1~T4 | 검증 연습을 위해 작성한 예시 | 원문과 다른 점이나 근거 부족을 판단 |
| 판단·근거·수정 답안 | 사람이 원문을 읽고 작성 | 표에 정리하고 판단 이유 설명 |

코드는 표를 선택하고 문자열·범주·누락 여부를 검사합니다. 원문의 의미를 이해해 자동으로 정답 판정을 내리는 코드는 아닙니다. 각 결과가 **데이터에서 읽힌 값인지, 코드가 계산한 값인지, 사람이 작성한 해설인지** 구분하는 것이 이번 실습의 출발점입니다.


## 실습 준비

아래는 **제공 코드**입니다. 그대로 실행하고 결과를 확인하세요. 함수 작성과 pandas 구문 자체는 평가하지 않습니다.

1. 세 파일의 크기, 컬럼, 자료형, 결측치와 빈 문자열을 확인합니다.
2. 상담 ID 수, 작업 유형별 행 수, 본문 글자 수를 확인합니다.
3. 상담 ID `40003`, `40005`를 연습 대상으로 준비합니다.
4. 선택한 상담의 세 파일 내 원문이 같은지 확인합니다.
5. 지시와 출력을 하나씩 조회할 수 있도록 보조 함수를 준비합니다.

`source_id` 하나에 여러 행이 있으므로 단순히 첫 행을 다른 작업의 결과로 사용하면 안 됩니다. 준비 코드가 추가하는 `record_id`는 **파일 이름 + 그 파일에서의 0부터 시작하는 행 위치**로 만든 실습용 식별자입니다. 원본 컬럼을 바꾸지 않으며, 파일의 행 순서가 바뀌면 이 식별자도 달라집니다.

본문 글자 수는 입력 규모를 살펴보기 위한 값입니다. **LLM의 토큰 수가 아니며**, 컨텍스트 윈도우 초과 여부를 직접 판정하는 값도 아닙니다.


In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(".")
classification = pd.read_csv(DATA_DIR / "엘지유플러스_분류.csv")
summarization = pd.read_csv(DATA_DIR / "엘지유플러스_요약.csv")
qa = pd.read_csv(DATA_DIR / "엘지유플러스_질의응답.csv")
tables = {"분류": classification, "요약": summarization, "질의응답": qa}

for name, table in tables.items():
    print(f"\n[{name}] 원본 {table.shape[0]:,}행 × {table.shape[1]}열")
    print(pd.DataFrame({"자료형": table.dtypes.astype(str),
                        "결측치": table.isna().sum()}).to_string())
    print("고유 상담 ID 수:", table["source_id"].nunique())
    print("빈 본문 수:", table["input"].str.strip().eq("").sum())
    print("빈 지시 수:", table["instruction"].str.strip().eq("").sum())
    print("빈 출력 수:", table["output"].str.strip().eq("").sum())
    print("작업 유형별 행 수:")
    print(table["task_category"].value_counts().to_string())
    print("행별 본문 글자 수(토큰 수가 아님):")
    print(table["input"].str.len().agg(["min", "mean", "max"]).round(2).to_string())
    table["record_id"] = [f"{name}-{i}" for i in range(len(table))]

practice_ids = [40003, 40005]
for source_id in practice_ids:
    texts = []
    print(f"\n[연습 상담 {source_id}]")
    for name, table in tables.items():
        part = table[table["source_id"] == source_id]
        print(name, "행 수:", len(part))
        texts.extend(part["input"].tolist())
    print("세 파일에서 이 상담의 원문이 모두 동일:", pd.Series(texts).nunique() == 1)
    print("이 상담 원문의 글자 수:", len(texts[0]))

def get_record(table, source_id, task_category, instruction=None):
    selected = table[(table["source_id"] == source_id)
                     & (table["task_category"] == task_category)]
    if instruction is not None:
        selected = selected[selected["instruction"] == instruction]
    if len(selected) != 1:
        raise ValueError("한 행을 선택해야 합니다. ID·작업 유형·질문을 다시 확인하세요.")
    return selected.iloc[0]

def show_record(row, include_input=False):
    print("\nrecord_id:", row["record_id"], " / source_id:", row["source_id"])
    print("작업 유형:", row["task_category"])
    print("지시 또는 질문:", row["instruction"])
    if include_input:
        print("상담 원문:\n" + row["input"])
    print("CSV에 저장된 출력:", row["output"])



[분류] 원본 16,080행 × 6열
                       자료형  결측치
source_id            int64    0
consulting_category    str    0
task_category          str    0
instruction            str    0
input                  str    0
output                 str    0
고유 상담 ID 수: 3216
빈 본문 수: 0
빈 지시 수: 0
빈 출력 수: 0
작업 유형별 행 수:
task_category
상담 요건    3216
상담 주제    3216
상담 사유    3216
상담 내용    3216
상담 결과    3216
행별 본문 글자 수(토큰 수가 아님):
min      670.00
mean    1583.42
max     2836.00

[요약] 원본 11,690행 × 6열
                       자료형  결측치
source_id            int64    0
consulting_category    str    0
task_category          str    0
instruction            str    0
input                  str    0
output                 str    0
고유 상담 ID 수: 3216
빈 본문 수: 0
빈 지시 수: 0
빈 출력 수: 0
작업 유형별 행 수:
task_category
기본 요약          3217
길이 제한 요약       2928
주제 중점 요약       2862
핵심 정보 추출 요약    2683
행별 본문 글자 수(토큰 수가 아님):
min      215.00
mean    1576.97
max     2836.00

[질의응답] 원본 7,497행 × 6열
                       자료형  결측치
source_id        

### 준비 코드 읽기 — 파일과 표의 구조부터 확인하기

`pandas`의 `DataFrame`은 행과 열로 이루어진 표입니다. 이번 데이터의 한 행은 ‘상담 한 개’ 자체보다 **특정 상담에 대한 작업 한 개**로 이해하면 좋습니다. 같은 상담에 분류·요약·질문을 여러 개 만들 수 있기 때문입니다.

| 코드·표현 | 하는 일 | 결과를 읽는 방법 |
|---|---|---|
| `Path(".")` | 현재 작업 폴더를 경로로 지정 | 데이터가 다른 폴더라면 해당 폴더로 수정 |
| `DATA_DIR / "엘지유플러스_분류.csv"` | 폴더와 파일 이름을 연결 | `/`는 여기서 경로 연결 역할 |
| `pd.read_csv(...)` | CSV를 표로 읽음 | 파일 조회이며 LLM 호출이 아님 |
| `tables = {"분류": classification, ...}` | 이름과 표를 짝지어 보관 | 세 파일에 같은 점검 절차를 적용하기 위함 |
| `tables.items()` | 이름과 표를 한 쌍씩 꺼냄 | 첫 반복에서 분류 표, 다음 반복에서 다른 표 확인 |
| `table.shape` | 표의 행 수·열 수 확인 | 행 수가 곧 고유 상담 수인 것은 아님 |
| `table.dtypes.astype(str)` | 열별 자료형 이름을 문자열로 표시 | 원본 상담 본문을 변환하는 작업이 아님 |
| `table.isna().sum()` | 열별 결측치 수 계산 | 비어 있는 값 확인 |
| `table["source_id"].nunique()` | 중복을 제외한 상담 ID 개수 계산 | 동일 ID의 작업 여러 행을 한 상담으로 셈 |
| `.str.strip().eq("").sum()` | 앞뒤 공백을 제외하면 빈 문자열인 값의 수 계산 | 결측치 검사와 별도로 내용 없는 문자열 확인 |
| `.value_counts()` | 작업 유형마다 몇 행인지 계산 | 상담 수보다 작업의 구성 확인에 초점 |
| `.str.len()` | 각 행의 본문 글자 수 계산 | 토큰 수를 계산하는 기능은 아님 |
| `.agg(["min", "mean", "max"])` | 길이의 최솟값·평균·최댓값 계산 | 파일에 있는 각 행을 기준으로 집계 |
| `.round(2)` | 계산값을 소수점 둘째 자리까지 반올림 | 본문을 자르거나 요약하는 기능이 아님 |

`print()`는 값을 화면에 보여 줍니다. `f"...{값}..."`는 문자열 중간에 값을 넣는 표기입니다. 예를 들어 `{table.shape[0]:,}`의 쉼표는 큰 행 수를 천 단위로 나누어 보여 줍니다. `.to_string(index=False)`는 표를 문자열로 출력하면서 왼쪽 인덱스를 숨기는 옵션이며, ID 열이나 실제 행을 삭제하지 않습니다.

### 같은 상담에 여러 작업이 있을 때의 ID

- `source_id`: 어느 상담에서 나온 자료인지 구분합니다.
- `task_category`: 그 상담에 대해 어떤 종류의 작업을 수행하는지 나타냅니다.
- `instruction`: 실제로 요구한 작업이나 질문 문장입니다.
- `record_id`: 현재 파일의 작업 행을 구별하기 위해 실습 코드가 추가한 이름입니다.

`[f"{name}-{i}" for i in range(len(table))]`는 현재 표의 각 행에 `분류-0`, `분류-1`처럼 이름을 붙입니다. `range(len(table))`는 0부터 행 수 직전까지의 번호를 만듭니다. 원본 표의 정렬 순서가 바뀌면 번호도 달라질 수 있으므로 원본 상담 ID와 같은 의미로 쓰지 않습니다.

준비 코드가 `record_id`를 추가한 뒤 표는 7열이 됩니다. 출력되는 ‘원본 6열’은 그 열을 추가하기 전에 확인한 값입니다. CSV 파일에 새 열을 저장하는 `to_csv()`는 실행하지 않습니다.


---
## 필수 1. 같은 원문과 서로 다른 분석 작업

### 문제 1-1. 분류·요약·정보 추출의 입력과 출력 비교

#### 문제 설명

상담 ID **40003**은 고객의 요금제 변경과 관련된 상담입니다. 같은 원문에 연결된 세 가지 작업을 비교합니다.

| 변수명 | 파일 | 선택할 task_category | 추가 선택 조건 |
|---|---|---|---|
| class_row | classification | 상담 내용 | 없음 |
| summary_row | summarization | 기본 요약 | 없음 |
| qa_row | qa | 의문사형 | instruction이 `고객의 결합 할인은 누구 앞으로 되어 있나요?` |

이번에는 저장된 지시와 출력을 읽습니다. 새로운 프롬프트를 설계하거나 모델을 호출하지 않습니다.

#### 요구사항

1. 제공된 `get_record()`로 위 세 행을 선택하세요.
2. `show_record(class_row, include_input=True)`로 상담 원문을 한 번 출력하고, 나머지 두 행은 지시와 출력만 확인하세요.
3. 세 행의 `input`이 동일한지 `class_row["input"] == summary_row["input"] == qa_row["input"]`로 확인하세요.
4. `작업`, `record_id`, `source_id`, `instruction`, `저장된_output`, `출력의 역할`을 담은 `task_comparison` 표를 만드세요. 출력의 역할은 본인이 읽고 작성합니다.
5. 선택한 분류 결과와 질의응답의 답을 뒷받침하는 원문 표현을 각각 하나 이상 적고, 요약에서는 요금제 변경 처리·결합 할인·멤버십에 관한 내용이 반영되었는지 확인하세요.

**보조 구문:** `get_record(표, ID, "작업 유형", "질문")`, `pd.DataFrame([{...}, {...}])`, `표.to_string(index=False)`

#### 해석 질문

**Q1.** 이 데이터에서 분석 대상, 작업 지시, 참고할 분류 범주, 저장된 출력은 어디에 있나요? 같은 `source_id`만으로 특정 출력 한 개를 식별할 수 있나요?

**Q2.** 세 작업의 결과는 각각 무엇이며 형태와 목적이 어떻게 다른가요? 분류 결과·질의응답 답변을 뒷받침하는 실제 원문과 요약의 핵심 내용을 설명하세요.

**Q3.** 주요 단어 추출, 의미가 유사한 상담 검색, 상담 내용의 분류·요약에는 TF-IDF·텍스트 임베딩·LLM 중 무엇을 각각 우선 고려할 수 있나요?

**Q4.** LLM의 텍스트 생성 흐름을 토큰·다음 토큰 예측·문맥을 포함해 설명하세요. 이번 코드의 `output` 조회가 그 생성 과정을 실행한 것인가요? 글자 수를 토큰 수라고 부를 수 있나요?

#### 제출 결과

- 세 행 조회, 원문 출력, 동일성 확인 코드
- `task_comparison` 표
- Q1~Q4 답변과 원문 근거


### 풀이 전 안내 — 같은 상담에서 정확한 작업 고르기

먼저 선택 조건표를 보고 파일·상담 ID·작업 유형을 하나씩 맞추세요. 질문이 여러 개일 수 있는 질의응답은 실제 질문 문장까지 지정합니다. `get_record()`가 한 행을 반환하면 `행["instruction"]`과 `행["output"]`으로 지시와 저장된 결과를 읽을 수 있습니다.

원문 동일성은 문자열 비교로 확인하고, 결과의 의미는 따로 읽어 판단하세요. 비교표의 ‘출력의 역할’에는 그 결과가 분류인지, 내용을 줄인 요약인지, 질문한 사실에 대한 답인지 자신의 말로 설명합니다. 원문에서 판단 근거가 되는 고객과 상담사의 발화를 함께 찾으세요.

같은 본문이어도 작업이 다르면 출력이 달라지는 것이 자연스럽습니다. 실제 LLM 생성 과정과 이번 코드의 파일 조회 과정을 구분해서 답하면 됩니다.


In [2]:
# 필수 1 코드를 작성하세요.
# 분류 파일에서 상담 ID 40003의 상담 내용 분류 예시로 선택
class_row = get_record(classification, 40003, '상담 내용')

# 요약 파일에서 같은 상담의 기본 요약 선택
summary_row = get_record(summarization, 40003, '기본 요약')

# 같은 상담의 특정 질문을 지정하여 선택
qa_row = get_record(qa, 40003, '의문사형', '고객의 결합 할인은 누구 앞으로 되어 있나요?')

# 2. 원문은 한 번만, 나머지는 지시·출력만
show_record(class_row, include_input=True)
show_record(summary_row)
show_record(qa_row)

# 3. 세 행의 input이 같은지 확인
print("input 동일 여부:",
      class_row["input"] == summary_row["input"] == qa_row["input"])

# 4. 비교 표
rows = [("분류", class_row, "직접 작성"),
        ("요약", summary_row, "직접 작성"),
        ("질의응답", qa_row, "직접 작성")]

task_comparison = pd.DataFrame([
    {
        "작업": name,
        "record_id": r["record_id"],
        "source_id": r["source_id"],
        "instruction": r["instruction"],
        "저장된_output": r["output"],
        "출력의 역할": role,
    }
    for name, r, role in rows
])
print(task_comparison.to_string(index=False))


record_id: 분류-13  / source_id: 40003
작업 유형: 상담 내용
지시 또는 질문: 이 상담 내용이 "일반 문의 상담", "업무 처리 상담", "고충 상담" 중에서 어디에 속하는지 알려 줘.
상담 원문:
상담사: 일상에 즐거운 변화 LGU+ <NAME>입니다.
고객: 아, 네, 안녕하세요? 저<DATE> 날 휴대폰을 바꿨는데요.
상담사: 네.
고객: 어, 제가 요금제를 그때 대리점에서는 <DATE> 요금제를 바꿀 수 있다고 했는데 제가 예전에 <PHONE_NUMBER> 통화했을 때는 <DATE> 바꿀 수 있다고 했거든요.
상담사: 아, 네.
고객: 네, 한번 확인을 좀 해주실 수 있을까요?
상담사: 네, 확인해 보겠습니다. 입력해 주신 번호가 <NAME> 고객님 본인 맞으십니까?
고객: 아, 네, 맞아요.
상담사: 감사합니다. <DATE>부터 변경 가능하신 날짜가 맞네요. 변경하셔도 되고요.
고객: 어, 네. 근데 그게 저한테 그 대리점에서 그 전에 바꾸면 위약금이 있다는 식으로 얘기했거든요, <DATE> 전에. 네.
상담사: 아, 네, <DATE>부터, 네.
고객: 가능한 거예요?
상담사: <DATE>부터 변경하셔도 위약금 안 나오는 기간 맞습니다.
고객: 아, 네.
상담사: 매장 쪽에서 이제 아무래도 기간 자체를 조금 넉넉하게 기준 잡다 보니까 하루 이틀 정도 뒤로 말씀을 드린 거 같은데요.
고객: 아, 네.
상담사: 네, <DATE>부터 변경하실 수 있는 게 맞아요.
고객: 아, 네, 그럼 혹시 5G 다이렉트 37.5라는 그 요금제도 제가 할 수 있는 건가요?
상담사: 어, 다이렉트 요금제인데요. 고객님, 근데 이게 조건이 휴대폰 기기를 구매하실 때 저희 U+샵 부서를 통해서 개통을 해주셔야 되고요. 그리고 일부 이런 결합 할인이나 약정이 저희가 들어가시면 안 되는 요금제이기 때문에, 네.
고객: 네, 아... 그러면 안 되네요.
상담사: 네, 맞습니다.
고객: 아, 그럼 혹시 5G 슬림+ 그 <CHARGE> 짜리는 가능해요?
상

### 필수 1 답변 작성란

**Q1.** 이 데이터에서 분석 대상, 작업 지시, 참고할 분류 범주, 저장된 출력은 어디에 있나요? 같은 `source_id`만으로 특정 출력 한 개를 식별할 수 있나요?
- 분석 대상은 input, 수행할 작업이나 질문은 instruction입니다.
- 선택한 분류 지시에는 일반 문의 상담, 업무 처리 상담, 고충 상담이라는 범주로 나눴다.
- 이처럼 범주와 작업 기준도 모델이  참고해야하는 정보다.
- consulting_category는 상담 분야 정보, task_category는 작업 유형

**Q2.** 세 작업의 결과는 각각 무엇이며 형태와 목적이 어떻게 다른가요? 분류 결과·질의응답 답변을 뒷받침하는 실제 원문과 요약의 핵심 내용을 설명하세요.
|작업|선택한 저장 결과 또는 핵심 내용|역할|
|---|---|---|
|분류|업무 처리 상담|주어진 범주 중 하나를 선택|
|요약|5G 슬림 + 변경 안내와 즉시 변경, 결합 할인 유지, 멤버십 등급 조정|주요 질문, 안내, 처리를 간결하게 정리|
|질의응답|남편|질문한 특정 사실에 답변|

- 분류 근거: 고객이 '네 그럼 바꿔주세요' 라고 요청한 부분이 있기 때문에 상담사가 '지금 즉시 변경을 해놓고 요금제 안내 사항을 문자 남겨드리겠습니다'라고 설명한다.
    <BR> -> 안내 뿐만 아니라 실제 변경 처리 상황을 보여주는 근거로 선택한 범주가 옳았는지 판단하는 근거가 된다.
- 질의 응답 근거 : 고객의 '아, 지금 저희 남편 앞으로 되어 있나요?'에 상담사가 '네, 맞습니다'라고 답변을 하기 때문에 누구에게 할인 지정이 되어 있는 지 확인한 문맥입니다.
- 요약 확인 : 요금제 변경 처리뿐 아니라 기존 결합 할인 유지, VIP 혜택을 더 이상 받지 못하고 일반 플러스 등급으로 조정된다는 안내가 반영되어  있다.
- 결합할인과 멤버십 혜택은 다른 내용이기 때문에 한쪽의 변경을 다른 쪽의 해지로 해석하는 것을 경계해야 한다.

**Q3.** 주요 단어 추출, 의미가 유사한 상담 검색, 상담 내용의 분류·요약에는 TF-IDF·텍스트 임베딩·LLM 중 무엇을 각각 우선 고려할 수 있나요?
- 특징적인 단어를 추출하려면 단어별 중요도를 계산하는 TF-IDF를 우선 고려할 수 있다.
- 의미가 유사한 상담들을 묶고 분류가 잘 됐는 지 보고 싶으면 임베딩과 유사도 검색을 우선 고려할 수 있다.
- 상담의 문맥과 지시를 바탕으로 분류 및 요약 결과를 생성하려면 LLM을 우선 고려할 수 있다.

**Q4.** LLM의 텍스트 생성 흐름을 토큰·다음 토큰 예측·문맥을 포함해 설명하세요. 이번 코드의 `output` 조회가 그 생성 과정을 실행한 것인가요? 글자 수를 토큰 수라고 부를 수 있나요?
- 교안의 생성 흐름은 입력 텍스트 -> 다음 토큰 예측 -> 토큰 생성 -> 반복 작업
- 토큰은 모델이 텍스트를 처리하는 기본 단위이며 단어, 단어의 일부, 문장 부호 등이 될 수 있다.
- 모델은 입력 문맥과 앞서 생성한 내용을 참고하여 다음 토큰의 확률을 예측하고 이어지는 응답을 만든다.
- 이는 정해진 답변을 데이터베이스에서 바로 찾아 반환하는 과정과 다르며 단어를 아무렇게나 무작위로 연결한다는 뜻도 아니다.


---
## 필수 2. 출력 변동과 분석 오류 검증

### 문제 2-1. 원문에 맞는 결과와 그럴듯한 오류 구분하기

#### 문제 설명

필수 1의 상담 **40003**을 계속 사용합니다. 다음 표들은 실제 모델 결과가 아니라 **검증 연습용으로 만든 출력 예시**입니다. 원문을 읽고 판단하세요.

검토할 요약의 범위는 **요금제 변경 처리와 기존 결합 할인 유지 여부**입니다.

| 예시 | 검토 대상 출력 |
|---|---|
| A | 고객 요청에 따라 5G 슬림+ 요금제로 변경했으며 기존 결합 할인은 유지되었다. |
| B | 기존 결합 할인은 그대로 두고, 고객이 요청한 5G 슬림+로 요금제를 바꾸었다. |
| C | 요금제 변경과 함께 기존 결합 할인도 해지되었다. |
| D | 상담사는 요금제 변경 보상으로 고객에게 현금 5만 원을 지급했다. |

같은 상담의 **상담 내용 분류**에서 사용할 범주는 필수 1의 지시에 있는 `일반 문의 상담`, `업무 처리 상담`, `고충 상담`입니다. 이 사례에서는 원문에서 요금제 변경을 처리했는지에 주목합니다.

| 분류 예시 | 출력 |
|---|---|
| F1 | 업무 처리 상담 |
| F2 | 일반 문의 상담 |
| F3 | 요금제 변경 처리 |

#### 요구사항

1. 요약 예시 A~D를 `summary_examples` 표로 만드세요. `예시`, `출력`, `판단`, `원문 근거 또는 확인 불가 이유`를 넣습니다. 판단은 원문을 읽고 직접 작성합니다.
2. A와 B의 문장이 동일한지 `==`로 확인하고, 표현 차이가 의미 오류인지 별도로 판단하세요.
3. 분류 예시 F1~F3를 `classification_examples`로 만들고, 허용 범주 목록 `allowed_categories`를 작성하세요. `classification_examples["출력"].isin(allowed_categories)`로 `허용 범주 여부`를 추가합니다.
4. 각 분류 예시의 `원문에 따른 판단`, `근거 또는 수정 방향`을 직접 작성하세요. 허용 범주 여부와 실제 분류 정확성을 구분합니다.
5. 오류 예시 C·D를 원문에 맞게 어떻게 고칠지 적으세요. C는 결합 할인 상태를 바로잡고, D는 확인 불가한 내용을 구분합니다.

#### 해석 질문

**Q1.** A와 B는 표현이 다르다는 이유만으로 오류인가요? 실제 LLM의 출력 변동성을 이번 예시로 측정했다고 말할 수 있나요?

**Q2.** C와 D의 문제는 각각 무엇인가요? 원문과의 불일치와 근거 없는 정보 생성을 구분하고, 각각의 수정 답안을 제시하세요.

**Q3.** F1~F3에서 허용된 범주를 사용한 경우와 원문에 적합한 경우를 구분하세요. 분류 결과가 일정한 형식이면 정확성도 보장되나요? F3 같은 표현이 저장되면 집계에 어떤 문제가 생기나요?

**Q4.** 근거 없는 정보, 다른 범주명, 필요한 정보의 누락이 발생했을 때 각각 무엇을 먼저 확인해야 하나요? 이러한 결과를 바로 저장하면 어떤 문제가 생길 수 있나요?

#### 제출 결과

- 요약·분류 예시 검토 표와 범주 여부 확인 코드
- C·D 수정 답안
- Q1~Q4 답변


### 풀이 전 안내 — 검사 결과의 뜻을 먼저 구분하기

A와 B를 `==`로 비교하면 문자가 같은지 알 수 있지만 의미가 같은지는 알 수 없습니다. 먼저 두 문장을 읽고, 이번 요약에서 요구한 핵심 사실이 무엇인지 나누어 확인하세요. 표현 순서가 달라진 것과 사실이 바뀐 것을 구분합니다.

분류의 `.isin(allowed_categories)`는 출력 문자열이 허용 목록에 있는지 확인합니다. 그 결과와 별개로 실제 상담에 맞는 범주인지 판단해야 합니다. 판단·근거 열은 원문을 읽고 직접 작성하세요.

오류를 고칠 때는 원문에서 확인할 수 있는 사실만 사용합니다. 틀린 상태를 바로잡는 경우와, 원문에 아예 없는 정보를 확인 불가로 구분하는 경우의 수정 방법이 다를 수 있습니다.


In [3]:
# 필수 2 코드를 작성하세요.
summaries = {
    "A": "고객 요청에 따라 5G 슬림+ 요금제로 변경했으며 기존 결합 할인은 유지되었다.",
    "B": "기존 결합 할인은 그대로 두고, 고객이 요청한 5G 슬림+로 요금제를 바꾸었다.",
    "C": "요금제 변경과 함께 기존 결합 할인도 해지되었다.",
    "D": "상담사는 요금제 변경 보상으로 고객에게 현금 5만 원을 지급했다.",
}

# 문자열 비교: 의미가 아니라 글자가 같은지만 알려줌
print("A == B:", summaries["A"] == summaries["B"])  # False

summary_check = pd.DataFrame([
    {"예시": "A", "요금제 변경": "반영(5G 슬림+)", "결합 할인": "유지",
     "판단": "정확", "근거": "원문 표현 직접 기입"},
    {"예시": "B", "요금제 변경": "반영(5G 슬림+)", "결합 할인": "유지",
     "판단": "정확", "근거": "A와 어순·표현만 다르고 사실은 같음"},
    {"예시": "C", "요금제 변경": "반영(요금제명 없음)", "결합 할인": "해지",
     "판단": "오류(사실 반전)", "근거": "원문은 유지 → 상태가 반대로 서술됨"},
    {"예시": "D", "요금제 변경": "언급만", "결합 할인": "누락",
     "판단": "오류(원문에 없는 정보)", "근거": "현금 보상 언급 없음, 범위 내 핵심 사실도 빠짐"},
])
print(summary_check.to_string(index=False))

allowed_categories = ["일반 문의 상담", "업무 처리 상담", "고충 상담"]

class_check = pd.DataFrame({
    "예시": ["F1", "F2", "F3"],
    "출력": ["업무 처리 상담", "일반 문의 상담", "요금제 변경 처리"],
})
class_check["허용_목록_포함"] = class_check["출력"].isin(allowed_categories)
class_check["판단"] = ["정확", "오류(범주 오선택)", "오류(형식 위반)"]
class_check["근거"] = [
    "실제로 요금제 변경을 처리함 → 업무 처리",
    "목록엔 있지만 단순 문의가 아니라 처리까지 이뤄짐",
    "내용은 맞는 방향이나 허용 범주가 아닌 자유 표현",
]
print(class_check.to_string(index=False))

A == B: False
예시      요금제 변경 결합 할인            판단                          근거
 A  반영(5G 슬림+)    유지            정확                 원문 표현 직접 기입
 B  반영(5G 슬림+)    유지            정확        A와 어순·표현만 다르고 사실은 같음
 C 반영(요금제명 없음)    해지     오류(사실 반전)        원문은 유지 → 상태가 반대로 서술됨
 D         언급만    누락 오류(원문에 없는 정보) 현금 보상 언급 없음, 범위 내 핵심 사실도 빠짐
예시        출력  허용_목록_포함         판단                          근거
F1  업무 처리 상담      True         정확     실제로 요금제 변경을 처리함 → 업무 처리
F2  일반 문의 상담      True 오류(범주 오선택) 목록엔 있지만 단순 문의가 아니라 처리까지 이뤄짐
F3 요금제 변경 처리     False  오류(형식 위반) 내용은 맞는 방향이나 허용 범주가 아닌 자유 표현


### 필수 2 답변 작성란

**Q1.** A와 B는 표현이 다르다는 이유만으로 오류인가요? 실제 LLM의 출력 변동성을 이번 예시로 측정했다고 말할 수 있나요?
- A와 B 문장 자체는 다르지만, 요청한 범위인 '5G 슬림+로 변경'과 '기존 결합 할인 유지'라는 핵심 내용을 모두 전달한다.
- 같은 핵심 의미를 원문에 맞게 표현한 요약이라면 단순한 표현 차이를 오류로 판단하지 않는다.
- 교안에서도 LLM은 동일 작업에서도 표현이 달라질 수 있다고 설명하고 있다.

**Q2.** C와 D의 문제는 각각 무엇인가요? 원문과의 불일치와 근거 없는 정보 생성을 구분하고, 각각의 수정 답안을 제시하세요.
- C는 원문 왜곡이  발생했다. 고객이 '그 할인은 그냥 기존대로 들어가는 거죠?'라고 묻자 상담사가 맞다고 답했고, 이후에도 요금제를 낮추더라도 할인이 유지 된다고 안내합니다.
    <br> -> 이를 해지라고 바꾼 것은 잘못된 내용이다.
    <br> -> 수정: '요금제를 변경했지만 기존 결합 할인은 유지되었다.'
- D는 근거 없는 정보 추가입니다. 원문에는 요금제 변경에 대한 현금 보상이나 5만원 지급을 언급한 적이 없습니다.
    <br> -> 수정: '원문에서 현금 보상 여부와 지급 금액은 확인할 수 없다' 
        <br> -> 요약에 불필요한 내용이라면 제거 가능하다.

**Q3.** F1~F3에서 허용된 범주를 사용한 경우와 원문에 적합한 경우를 구분하세요. 분류 결과가 일정한 형식이면 정확성도 보장되나요? F3 같은 표현이 저장되면 집계에 어떤 문제가 생기나요?
|예시| 허용 범주인가?|이 상담의 적절한 분류인가?|이유|
|---|---|---|---|
|F1 업무 처리 상담|예|예|고객 요청에 따라 요금제 변경을 실제 처리함|
|F2 일반 문의 상담|예|아니오|허용 문자열이지만 실제 변경 처리까지 있는 이 사례의 분류와 맞지 않음|
|F3 요금제 변경 처리|아니오|지정 범주로 수정 필요|처리 의미는 전달하지만 허용된 범주명이 아님|
- F2 처럼 형식이 맞더라도 분류가 틀릴 수 있다.
- F3 처럼 비슷한 의미의 다른 표현을 그대로 저장하면 업무 처리 상담과 별도 범주로 집계될 수 있다.
- 반드시 원문과 확인하여 범주명으로 정리 과정이 필요하다.

**Q4.** 근거 없는 정보, 다른 범주명, 필요한 정보의 누락이 발생했을 때 각각 무엇을 먼저 확인해야 하나요? 이러한 결과를 바로 저장하면 어떤 문제가 생길 수 있나요?
- 원문에 없는 정보를 생성했다면 입력에 그 사실이 존재하는지 확인한다. 없으면 내용을 만들어 채우지 않는다.
- 같은 의미를 다른 범주명으로 냈다면 출력할 범주와 판단 기준이 명확한지 확인한다.
- 필요한 정보가 누락되었다면 요청할 항목이 명확한지, 원문에 해당 정보가 있는지, 입력, 출력이 모두 전달되었는 지 확인한다.


---
## 과제 1. 다른 상담의 결과를 원문으로 검증

### 문제 3-1. 통화 중 대기 상담의 정보·분류·누락 확인

#### 문제 설명

이번에는 상담 **40005**를 사용합니다. 필수 문제에서 수행한 조회와 검증을 같은 수준으로 적용합니다.

| 변수명 | 파일 | task_category | 추가 선택 조건 |
|---|---|---|---|
| task_class_row | classification | 상담 내용 | 없음 |
| task_summary_row | summarization | 길이 제한 요약 | 없음 |
| task_qa_row | qa | 예/아니요형 | instruction이 `고객은 지금 추가 결합 할인을 받겠다고 했어?` |

다음은 이 상담에 대한 **학습용 검토 문장**입니다. 실제 모델 응답이 아닙니다.

| 항목 | 검토 문장 |
|---|---|
| T1 | 상담사는 통화 중 대기 서비스를 삭제한 뒤 다시 신청하고 전화로 확인했다. |
| T2 | 고객은 상담 중 추가 결합 할인 가입을 즉시 완료했다. |
| T3 | 통화 중 대기 문제의 원인은 휴대폰 기기 고장으로 확인되었다. |
| T4 | 고객은 추후 상담사의 연락을 받는 데 동의했다. |

추가로 40003과 40005에 대해 **상담 내용 분류를 각각 한 번씩 요청했지만**, 아래 한 개의 결과만 돌아왔다고 가정합니다.

| source_id | task_category | 가상 반환 결과 |
|---:|---|---|
| 40003 | 상담 내용 | 업무 처리 상담 |

#### 요구사항

1. 지정한 세 행을 선택하세요. `task_class_row`에서 원문 전체를 한 번 출력하고, 나머지 행의 지시와 저장된 출력을 확인하세요.
2. T1~T4에 대해 `항목`, `판단`, `원문 근거 또는 확인 불가 이유`를 담은 `task_validation` 표를 작성하세요.
3. 허용 범주 `일반 문의 상담`, `업무 처리 상담`, `고충 상담` 중 원문에 적합한 분류를 정하고 근거를 쓰세요. 실제 서비스 재설정 처리가 있었는지에 주목합니다.
4. 상담을 **2문장, 300자 이내**로 요약하고, 선택한 질의응답의 질문에 답하세요. 지금 할인 가입을 완료했는지와 추후 연락에 동의했는지를 구분합니다. 사람의 예시 답안임을 표시하세요.
5. 요청 ID 목록 `[40003, 40005]`와 위 가상 반환 표를 만든 뒤, 반환되지 않은 ID를 확인하세요. 제공 구문을 사용해도 됩니다: `[sid for sid in requested_ids if sid not in returned["source_id"].tolist()]`. 이번 비교에서는 모든 요청이 같은 작업이고 각 ID당 한 번만 요청했다고 가정합니다.

#### 해석 질문

**Q1.** T1~T4 중 원문에 근거한 내용, 원문과 다른 내용, 근거를 확인할 수 없는 내용은 각각 무엇인가요? 실제 표현을 근거로 설명하세요.

**Q2.** 적절한 분류, 2문장 요약, 질의응답 답안을 제시하세요. “추후 상담사 연락에 동의”를 “추가 할인 가입 완료”로 바꾸면 안 되는 이유는 무엇인가요?

**Q3.** 가상 반환 결과에서 누락된 상담 ID는 무엇인가요? ID·작업 연결이 필요한 이유와, 많은 상담을 한 번에 LLM에 넣을 때 고려해야 할 한계를 설명하세요.

**Q4.** 원문의 `<NAME>`, `<DATE>`, `<CHARGE>`를 실제 값으로 채워도 되나요? 외부 LLM API에 업무 데이터를 전달하기 전 확인할 사항은 무엇인가요? 원문에 없는 고장 원인을 지시 수정만으로 확정할 수 있나요?

#### 제출 결과

- 세 행 조회와 원문 출력 코드
- `task_validation`, 분류·요약·질의응답의 직접 작성 답안
- 가상 반환 결과의 누락 ID 확인 코드와 결과
- Q1~Q4 답변


### 풀이 전 안내 — 조치와 원인, 현재와 추후를 구분하기

상담 전체를 읽으면서 ‘어떤 조치를 했는지’, ‘어떤 결과를 확인했는지’, ‘고객이 무엇에 동의했는지’를 나누어 적어 보세요. 서비스를 다시 설정했다는 사실만으로 문제가 생긴 원인까지 확정하지 않습니다. 고객의 짧은 동의 응답은 앞선 제안과 함께 읽습니다.

요약은 핵심 내용을 먼저 정리하고, 두 문장인지 직접 확인한 뒤 `len(요약문)`으로 300자 이내인지 확인하세요. 글자 수가 맞아도 사실이 바뀌거나 필요한 조건이 빠지지 않았는지 다시 읽어야 합니다.

누락 검사는 요청 ID마다 반환 ID 목록에 들어 있는지 비교합니다. 이 과제의 가상 요청은 같은 작업을 각 ID에 한 번씩 요청한 경우입니다. 실제 LLM을 호출하거나 누락 원인을 실험하는 과제가 아닙니다.


In [ ]:
# 과제 1 코드를 작성하세요.


### 과제 1 답변 작성란

**Q1.**

- 답변:

**Q2.**

- 답변:

**Q3.**

- 답변:

**Q4.**

- 답변:


## 개념 정리 — 이번 강의에서 구분할 것

| 구분 | 첫 번째 의미 | 두 번째 의미 |
|---|---|---|
| source_id / record_id | 어느 상담인지 구별 | 현재 파일의 어느 작업 행인지 구별 |
| task_category / output | 수행할 작업의 종류 | 그 작업에 대한 저장된 결과 |
| 원문 / 작업 지시 | 무엇을 분석하는지 | 무엇을 하도록 요청하는지 |
| 저장된 출력 조회 / LLM 생성 | 파일에 있던 문자열을 읽음 | 문맥을 바탕으로 토큰을 생성 |
| 글자 수 / 토큰 수 | 문자열의 길이 | 모델이 처리하는 단위의 개수 |
| 작업 차이 / 출력 변동성 | 다른 작업을 요청해서 출력이 달라짐 | 동일한 작업에서도 표현이나 결과가 달라질 수 있음 |
| 문자열 동일성 / 의미 정확성 | 글자와 순서가 같은지 | 원문의 사실과 요청에 맞는지 |
| 허용 범주 여부 / 분류 정확성 | 목록 안의 이름인지 | 실제 상담에 적절한 범주인지 |
| 원문 왜곡 / 근거 없는 정보 | 확인된 사실을 다르게 표현 | 원문에 없는 사실을 추가 |
| 결과 포함 여부 / 결과 정확성 | 요청한 작업의 결과가 돌아왔는지 | 돌아온 내용이 맞는지 |

답안을 작성할 때는 ‘출력에서 무엇을 주장했는가 → 원문에서 무엇을 확인했는가 → 일치·불일치·확인 불가 중 어떻게 판단하는가’ 순서로 설명해 보세요. 근거 문장만 붙이기보다 그 문장이 왜 판단을 뒷받침하는지 적으면 좋습니다.


---
## 실습 마무리

다음 질문에 자신의 말로 답하세요.

1. 이번 실습의 분석 문제는 무엇이었나요?
2. 같은 상담을 분류·요약·질의응답으로 다루었을 때 결과가 어떻게 달랐나요?
3. 결과가 맞는지 무엇을 근거로 판단했나요?
4. 문자열 비교·허용 범주 확인만으로 해결되지 않는 검증은 무엇인가요?
5. 실제 LLM을 활용할 때 입력 크기·문서 연결·개인정보 측면에서 무엇을 확인해야 하나요?


### 실습 마무리 답변 작성란

1. 이번 실습의 분석 문제는 무엇이었나요?
- 두 상담을 이용하여 같은 원문에서 필요한 결과를 구분하고, 출력이 사실과 지시를 지켰는 지 확인했다.
- 상담 40003에서는 분류, 요약, 할인 지정 대상을 비교했고, 상담 40005에서는 서비스 조치와 고객의 동의 범위, 결과 누락을 검토했다.

2. 같은 상담을 분류·요약·질의응답으로 다루었을 때 결과가 어떻게 달랐나요?
- 분류는 지정된 범주 하나를 선택하고, 요약은 주요 내용을 짧게 정리하며, 선택한 질의응답은 특정 사실에 답하기 때문이다.
- 세 작업의 원문이 같다는 True의 결과는 비교 대상을 확인해줄 수 있다.
- 원하는 출력까지 같아야 한다는 뜻은 아니다.

3. 결과가 맞는지 무엇을 근거로 판단했나요?
- 자연스러운 말투나 저장된 답변이라는 이유로 받아들이지 않고, 원문과 지시를 읽었다.
- 40003의 즉시 변경 처리와 결합 할인 유지 발화
- 40005의 서비스 재설정과 추후 연락 동의를 각각 근거로 삼았다.
- 원문에 없는 보상금, 고장 원인 등은 확인불가로 구분하여야 한다.

4. 문자열 비교·허용 범주 확인만으로 해결되지 않는 검증은 무엇인가요?
- 코드는 ID, 작업 조건에 맞는 행을 선택하고, 본문이나 요약 문장이 같은 지 비교하고, 범주가 목록에 있는 지 검사하고, 누락 ID를 찾았다.
- A와 B의 비교는 False, F1~3까지의 범주 검사는 True, True, False, 누락 ID는 [40005]이다.
- 이 값들은 각각 문자열 비교, 목록 포함 여부의 결과이다.

5. 실제 LLM을 활용할 때 입력 크기·문서 연결·개인정보 측면에서 무엇을 확인해야 하나요?
- 입력과 생성 내용이 모델의 처리 범위에 들어가는지, 출력이 어떤 상담의 어느 작업에 대응하는지
- 필요한 결과가 빠지지 않았는지 확인해야한다.
- 업무 자료를 전달할 때는 개인정보와 활용 권한, 처리 정책도 확인해야한다.
- 다만 이번 결과를 실제 모델의 정확도나 출력변동성에 대한 실험 결과로 설명하지는 않는다.
